# Trend distortion: does debiasing/downscaling change how scenarios compare?

This notebook evaluates whether bias correction and spatial disaggregation (BCSD) change how
scenarios compare to one another, relative to what the raw GCM itself shows. "Distortion" means a
*difference of differences*: how much a scenario-to-scenario delta, for example `G6-1.5K minus
SSP245` averaged over a multi-decade window, moves once debiasing or downscaling is applied. For each
comparison we compute the period-mean delta at three points in the pipeline (raw GCM, coarse
debiased, and downscaled debiased recoarsened back onto the raw grid so it is comparable cell for
cell) and flag grid cells where the debiased or downscaled delta departs from the raw delta by more
than a tolerance, in both absolute and percent terms.

This is a **rough** check that does no statistical significance testing. A multi-decade mean
difference between two scenarios at a single grid cell reflects both the forced scenario signal and
internal variability, and nothing here attempts to separate the two, so some of what gets flagged may
be noise in a single-member comparison rather than a real change in how the pipeline represents the
scenario signal. Treat flagged cells as places to look more closely, not as proof of a bug.

Every comparison is enumerated from the store rather than written by hand, and `srm.lineage` resolves
each scenario leaf's baseline member. That is what lets the notebook cover every variable at once
instead of one: the correct SSP245 bridge member for the SAI comparison differs by variable (issue
#448), so one hardcoded member cannot serve `tas` and `tasmax` together. On the v0.12.0 store this
yields 25 comparisons, of which 22 can be evaluated. The three `dtr` comparisons cannot, because
`dtr` is derived inside the pipeline and the raw GCM carries no `dtr` variable to difference against.

### What this notebook produces

| Output | Path under `QAQC_DIR` | Contents |
| --- | --- | --- |
| Summary table | `trend_distortion_summary.csv` | one row per comparison and stage pair |
| Per-comparison table | `trend_distortion_{comparison_id}.csv` | the three rows for one comparison |
| Per-comparison figure | `distortion_{comparison_id}.png` | absolute and percent distortion, three stage pairs |
| Overview figure | `overview_{variable}.png` | raw delta, downscaled delta, and distortion across families |

Each comparison is measured three ways, all with the same both-tolerance rule.

| Stage pair | Question it answers |
| --- | --- |
| `coarse_debiased_vs_raw` | did debiasing distort the delta? |
| `downscaled_debiased_vs_raw` | did debiasing and downscaling together distort it? |
| `downscaled_debiased_vs_coarse_debiased` | did downscaling distort it further, beyond debiasing alone? |

`frac_area_sign_flip` asks something stricter than any of those. Rather than how large the distortion
is, it asks whether the two stages disagree on the *direction* of the change at a cell, which is the
case where "does SAI increase or decrease this variable here" depends on which pipeline stage you
read it from.

In [1]:
import functools
import os
import pathlib
import time

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import coiled
import dask
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import zarr
from icechunk.xarray import to_icechunk

from srm import catalog
from srm.cache import ArtifactCache
from srm.config import ClusterConfig, _ensure_root_group, _icechunk_storage_for_path
from srm.downscaling_utils import interpolate_fine_to_coarse_grid
from srm.qaqc import (
    DISTORTION_FAMILIES,
    DISTORTION_STAGE_PAIRS,
    DISTORTION_STAGES,
    area_weights,
    calculate_distortion_flags,
    compute_deltas,
    distortion_fields,
    distortion_summary,
    enumerate_scenario_comparisons,
    sign_flip_mask,
)

os.environ["FRISKY_SUMMARY"] = "off"
from frisky import hijack  # noqa: E402  # after FRISKY_SUMMARY, which it reads at import

# NOTE: this notebook deliberately does NOT call dask_array.xarray.register(). Frisky's
# query-optimized array backend is incompatible with icechunk's distributed write path, which this
# notebook depends on to cache the period-mean grids. icechunk builds its session-merge step with
# classic `dask.array`, so a `dask_array` collection reaching `da.reduction` gets re-wrapped and
# `icechunk.distributed.extract_session` then fails with "AttributeError: 'function' object has no
# attribute 'session'". The backend is global once registered, so the write cannot be worked around
# locally. Frisky's client (`hijack` below) is unaffected and still used.

zarr.config.set({"async.concurrency": 128})

# --- Run parameters --------------------------------------------------------
GCM = "CESM2-WACCM"
STORE_URI = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/CESM2-WACCM-ERA5-global.icechunk"
BRANCH = "v0.12.0"
VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds", "hurs"]

# The two multi-decade windows every delta is built from. Scenario minus historical compares
# mean(SLICE_FUT) against mean(SLICE_HIST); G6 minus SSP245 compares the two over SLICE_FUT.
# SLICE_FUT starts well after 2035, which matters because pre-2035 g6_1p5k is bridged from SSP245
# (issue #448) and a window reaching into the bridge would compare a scenario against itself.
SLICE_HIST = slice("1978-01-01", "2014-12-31")
SLICE_FUT = slice("2055-01-01", "2085-12-31")

# A stage pair is flagged when more than this fraction of the (area-weighted) domain is distorted.
FLAG_THRESHOLD = 0.02

# How this run gets its numbers.
#
#   "full"   recompute the period-mean grids from source. Reads ~1.6 TB and needs the cluster.
#   "cached" read the grids a previous "full" run committed. Needs no cluster at all.
#   "auto"   use "cached" when every grid this notebook needs is committed, else "full".
#
# Everything after Part 1 is arithmetic on ~120 MB of cached grids, so re-picking a tolerance,
# adding a statistic, or redrawing every figure is a cached-mode edit that costs seconds.
COMPUTE_MODE = "auto"

# Figures are always saved; SHOW_FIGURES controls what renders inline. Twenty-five six-panel cartopy
# figures inline would push this notebook past 100 MB, so the default shows only the comparisons that
# cross FLAG_THRESHOLD. Accepts "flagged", "all", "none", or a list of comparison ids.
QAQC_DIR = "/scratch/synced/qa_plots/trend_distortion/"
SAVE_FIGURES = True
SHOW_FIGURES = "flagged"
OVERVIEW_VARIABLES = ["pr", "tas","tasmax","tasmin","rsds"]

# Optional fast path for a regional test run: (lat_slice, lon_slice), or None for the whole globe.
SUBSET = None


def apply_subset(da: xr.DataArray) -> xr.DataArray:
    """Restrict to SUBSET for a fast regional run; identity when SUBSET is None."""
    if SUBSET is None:
        return da
    lat_slice, lon_slice = SUBSET
    return da.sel(lat=lat_slice, lon=lon_slice)


# Artifact cache, named and branched the way the pipeline names its own stores so that versioning
# stays a matter of icechunk branches rather than path segments. The subset id comes from
# ArtifactCache so the two conventions cannot drift.
CACHE_SUBSET_BOUNDS = (
    None if SUBSET is None else (SUBSET[0].start, SUBSET[0].stop, SUBSET[1].start, SUBSET[1].stop)
)
CACHE_SUBSET_ID = ArtifactCache._get_subset_id(CACHE_SUBSET_BOUNDS)
CACHE_STORE_URI = f"s3://carbonplan-scratch/srm/qaqc/trend-{GCM}-ERA5-{CACHE_SUBSET_ID}.icechunk"

## Per-variable units and tolerances

One table drives every variable-specific number in the notebook, so a threshold lives in exactly one
place. `scale` converts a stored period mean into its reporting unit and is applied once, on the way
out of the cache, which puts every delta, flag, statistic, and axis downstream on the same scale as
the tolerances.

Daily series carry a *different* factor from period means. For `pr` a period mean reads naturally as
mm/yr (86400 seconds times 365 days) while a daily value reads as mm/day (86400 seconds), and using
the annual factor on a daily series would inflate it 365-fold. The two are separate columns for that
reason.

The absolute tolerance is a few percent of the variable's typical scenario signal, and the
downscaling-only percent tolerance is tighter than the raw one because downscaling should not change
a coarse-scale delta at all, only redistribute it inside a coarse cell. A percent tolerance of `0`
disables the percent condition rather than applying it literally, which is the intended setting for
temperature in kelvin where a 1 K change on a 300 K mean reads as 0.3 %.

The `rsds` and `hurs` rows were reasoned from the expected size of the signal rather than measured,
and the calibration table further down now reports how they landed. `hurs` at 0.5 % sits near the 85th
percentile of its own distortion, which is a usable gate. `rsds` at 0.5 W m-2 sits near the 48th, so
it admits more than half the map and the percent tolerance ends up deciding the flag by itself.
Precipitation inherits the same problem from the original notebook, with 10 mm/yr sitting at the 57th
percentile. Raising those two would make the absolute condition mean something again, and the
percentile columns say where to put them.

The `dtr` row is currently unused. `dtr` is derived during the pipeline rather than read from the GCM,
so the raw catalog has no `dtr` variable and there is no raw delta for a debiased one to be compared
against. Part 1 reports it as failed and drops its three comparisons, which is the intended
behaviour: a variable that cannot be checked is better named than silently omitted.

In [5]:
_SETTINGS_COLUMNS = (
    "units",
    "scale",
    "daily_units",
    "daily_scale",
    "abs_tol",
    "pct_tol",
    "pct_tol_downscaling",
    "sign_flip",
    "cmap",
)
_SETTINGS_ROWS = [
    ("tas", "K", 1.0, "K", 0.25, 0.0, 0.0, 0.0, 0.25, "RdBu_r"),
    ("tasmax", "K", 1.0, "K", 0.25, 0.0, 0.0, 0.0, 0.25, "RdBu_r"),
    ("tasmin", "K", 1.0, "K", 0.25, 0.0, 0.0, 0.0, 0.25, "RdBu_r"),
    ("pr", "mm/yr", 86400.0 * 365, "mm/day", 86400.0 * 10, 5.0, 2.0, 0.5, 5.0, "BrBG"),
    ("rsds", "W m-2", 1.0, "W m-2", 3.0, 0.25, 0.25, 0.5, 0.5, "PuOr"),
    ("hurs", "%", 1.0, "%", 5.0, 1.0, 2.0, 0.5, 0.5, "BrBG"),
]
VARIABLE_SETTINGS = {
    row[0]: dict(zip(_SETTINGS_COLUMNS, row[1:], strict=True)) for row in _SETTINGS_ROWS
}

# The stage pair that isolates downscaling from debiasing, and so takes the tighter percent tolerance.
DOWNSCALING_ONLY = "downscaled_debiased_vs_coarse_debiased"


def settings(var: str) -> dict:
    """Reporting units and tolerances for a variable."""
    if var not in VARIABLE_SETTINGS:
        raise KeyError(f"no distortion settings for {var!r}; add a row to _SETTINGS_ROWS above")
    return VARIABLE_SETTINGS[var]


def tolerances(var: str, stage_pair: str) -> tuple[float, float]:
    """(absolute, percent) tolerance for one variable and stage pair."""
    entry = settings(var)
    pct = entry["pct_tol_downscaling"] if stage_pair == DOWNSCALING_ONLY else entry["pct_tol"]
    return entry["abs_tol"], pct


print(pd.DataFrame.from_dict(VARIABLE_SETTINGS, orient="index").to_string())

        units       scale daily_units  daily_scale  abs_tol  pct_tol  pct_tol_downscaling  sign_flip    cmap
tas         K         1.0           K         0.25     0.00     0.00                  0.0       0.25  RdBu_r
tasmax      K         1.0           K         0.25     0.00     0.00                  0.0       0.25  RdBu_r
tasmin      K         1.0           K         0.25     0.00     0.00                  0.0       0.25  RdBu_r
pr      mm/yr  31536000.0      mm/day    864000.00     5.00     2.00                  0.5       5.00    BrBG
rsds    W m-2         1.0       W m-2         3.00     0.25     0.25                  0.5       0.50    PuOr
hurs        %         1.0           %         5.00     1.00     2.00                  0.5       0.50    BrBG


## The output store and comparison enumeration

One icechunk store holds every result, versioned by branch. A comparison needs its variable and
member present at all three pipeline stages, so we discover the fine-grid leaves under the top-level
scenario groups and the coarse-grid leaves under `debiased_coarse/`, then keep the intersection. A
leaf present at one resolution but not the other is reported rather than silently dropped.

`enumerate_scenario_comparisons` then resolves each scenario leaf's baseline member through
`srm.lineage` and returns one row per comparison it can build, plus the ones it cannot and why. A
missing baseline leaf or an unregistered lineage combination becomes a skip with a reason, never a
fabricated pairing.

In [6]:
repo = icechunk.Repository.open(_icechunk_storage_for_path(STORE_URI))
tree = xr.open_datatree(repo.readonly_session(BRANCH).store, engine="zarr", chunks={})

SCENARIO_GROUPS = sorted({group for pair in DISTORTION_FAMILIES.values() for group in pair})


def _node(path: str):
    """Datatree node at a slash-separated path, or None when any segment is absent."""
    node = tree
    for part in path.split("/"):
        if part not in node.children:
            return None
        node = node[part]
    return node


def discover_leaves(root: str | None = None) -> set[tuple[str, str, str]]:
    """Every (scenario, variable, member) leaf under the top level or under `root`."""
    found = set()
    for group in SCENARIO_GROUPS:
        node = _node(group if root is None else f"{root}/{group}")
        if node is None:
            continue
        for var in node.children:
            if var in VARIABLES:
                found.update((group, var, member) for member in node[var].children)
    return found


fine_leaves = discover_leaves()
coarse_leaves = discover_leaves("debiased_coarse")
leaves = fine_leaves & coarse_leaves

print(f"{len(fine_leaves)} fine-grid leaves, {len(coarse_leaves)} coarse-grid leaves")
for label, gap in [
    ("fine only (no debiased_coarse counterpart)", fine_leaves - coarse_leaves),
    ("coarse only (no downscaled counterpart)", coarse_leaves - fine_leaves),
]:
    if gap:
        print(f"  {label}: {sorted('/'.join(key) for key in gap)}")
print(f"{len(leaves)} leaves usable at all three stages")

22 fine-grid leaves, 22 coarse-grid leaves
22 leaves usable at all three stages


In [7]:
comparisons, skipped = enumerate_scenario_comparisons(leaves, gcm=GCM, variables=VARIABLES)

print(f"{len(comparisons)} comparisons enumerated:")
print(
    comparisons.drop(columns=["comparison_id"]).to_string(index=False)
    if len(comparisons)
    else "  none"
)
if len(skipped):
    print(f"\nSkipped {len(skipped)}:")
    print(skipped.to_string(index=False))

22 comparisons enumerated:
  family variable after_scenario after_member before_scenario before_member
ssp_hist     hurs         ssp245          003      historical      r3i1p1f1
ssp_hist     hurs         ssp245          008      historical      r3i1p1f1
ssp_hist       pr         ssp245          003      historical      r3i1p1f1
ssp_hist       pr         ssp245          008      historical      r3i1p1f1
ssp_hist     rsds         ssp245          003      historical      r3i1p1f1
ssp_hist     rsds         ssp245          008      historical      r3i1p1f1
ssp_hist      tas         ssp245          003      historical      r3i1p1f1
ssp_hist      tas         ssp245          008      historical      r3i1p1f1
ssp_hist   tasmax         ssp245          008      historical           001
ssp_hist   tasmin         ssp245          008      historical           001
 g6_hist     hurs        g6_1p5k          003      historical      r3i1p1f1
 g6_hist       pr        g6_1p5k          003      historical

## The artifact cache

Everything this notebook reports is arithmetic on two-dimensional period-mean grids: one per
(pipeline stage, scenario, variable, member). Producing them is the whole expensive part of the run,
and they are small enough that persisting all of them is free.

| Stage | Grid | Per grid | Read to produce it |
| --- | --- | --- | --- |
| `raw` | 192 x 288 | 0.22 MB | 6.9 GB per member, with 4x chunk amplification |
| `coarse_debiased` | 192 x 288 | 0.22 MB | 6.9 GB per leaf |
| `downscaled_debiased` | 721 x 1440 | 4.15 MB | 128.9 GB per leaf |
| `coarsened_downscaled_debiased` | 192 x 288 | 0.22 MB | derived from the fine grid |

For the v0.12.0 store that is roughly 1.6 TB read and about 120 MB cached. Measured on this run, 88 of
the 100 required grids computed in 9 minutes 42 seconds of wall time on 30 workers; the 12 that did
not are `dtr`, which has no raw stage to read. Once the grids are committed, choosing tolerances,
adding a summary statistic, and redrawing every figure all happen in seconds with no cluster, which is
what makes the tolerances above tractable to refine.

Grids are cached in the store's own units and scaled on the way out, so changing a reporting unit
does not invalidate the cache. One commit per variable keeps a crashed run from leaving a partial set
behind: an icechunk commit is atomic, so a write becomes visible only when it lands and no completion
marker is needed to tell a finished artifact from an abandoned one.

In [8]:
cache_repo = icechunk.Repository.open_or_create(_icechunk_storage_for_path(CACHE_STORE_URI))
if BRANCH not in cache_repo.list_branches():
    cache_repo.create_branch(BRANCH, cache_repo.lookup_branch("main"))
# A branch cut from the root snapshot carries no root group, so the first writers to two different
# groups would each try to create it and their commits could not rebase. See issue #521.
_ensure_root_group(cache_repo, BRANCH)


def mean_group(key: tuple[str, str, str, str]) -> str:
    """Cache group for one (stage, scenario, variable, member) period-mean grid."""
    return "means/" + "/".join(key)


def as_grid(da: xr.DataArray) -> xr.DataArray:
    """Drop everything but lat/lon.

    A period mean keeps whatever scalar coordinates its source carried, such as the selected
    ensemble member or a leftover dayofyear. Stripping them is what lets grids from three different
    stages be stored alike and then subtracted without xarray aligning on a coordinate that means
    nothing here.
    """
    return da.drop_vars([c for c in da.coords if c not in {"lat", "lon"}], errors="ignore")


def cached_groups() -> set[str]:
    """Every group currently committed on the cache branch.

    This is the cache's only existence test and it is exact rather than approximate: a write becomes
    visible only when its commit lands, so a run that died mid-write leaves nothing here to mistake
    for a finished grid.
    """
    session = cache_repo.readonly_session(BRANCH)
    cached_tree = xr.open_datatree(session.store, engine="zarr", chunks={})
    return {str(group).lstrip("/") for group in cached_tree.groups}


def save_period_means(means: dict, label: str) -> str:
    """Commit a batch of already-computed period-mean grids in one snapshot."""
    session = cache_repo.writable_session(BRANCH)
    for key, grid in means.items():
        stage, scenario, var, member = key
        out = as_grid(grid).rename("mean").to_dataset()
        # Carry the identity in attrs so a load can rebuild keys without parsing group names.
        out.attrs.update(stage=stage, scenario=scenario, variable=var, ensemble_member=member)
        to_icechunk(out, session, group=mean_group(key), mode="w")
    return session.commit(f"cache {len(means)} period-mean grid(s) for {label}")


def load_period_means(keys) -> dict:
    """Read cached period-mean grids into memory, keyed the way Part 1 produces them."""
    groups = cached_groups()
    missing = [key for key in keys if mean_group(key) not in groups]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} period-mean grid(s) missing from {CACHE_STORE_URI} on branch {BRANCH}, "
            f"first few: {[mean_group(key) for key in missing[:3]]}. Run this notebook once with "
            "COMPUTE_MODE='full' on this branch to produce them."
        )
    session = cache_repo.readonly_session(BRANCH)
    return {key: xr.open_zarr(session.store, group=mean_group(key))["mean"].load() for key in keys}


def required_grid_keys(frame: pd.DataFrame) -> list[tuple[str, str, str, str]]:
    """Every grid the enumerated comparisons need, deduplicated across comparisons.

    A leaf appearing in several comparisons, as SSP245 does when it serves both as the future half of
    a scenario-minus-historical delta and as the baseline of the SAI delta, is reduced once.
    """
    keys = set()
    for row in frame.itertuples():
        for scenario, member in (
            (row.after_scenario, row.after_member),
            (row.before_scenario, row.before_member),
        ):
            keys.update((stage, scenario, row.variable, member) for stage in DISTORTION_STAGES)
    return sorted(keys)


REQUIRED_KEYS = required_grid_keys(comparisons)
_cached = cached_groups()
_missing = [key for key in REQUIRED_KEYS if mean_group(key) not in _cached]

print(f"Cache: {CACHE_STORE_URI} (branch {BRANCH})")
print(f"  grids required: {len(REQUIRED_KEYS)}")
print(f"  grids cached:   {len(REQUIRED_KEYS) - len(_missing)}")

Cache: s3://carbonplan-scratch/srm/qaqc/trend-CESM2-WACCM-ERA5-global.icechunk (branch v0.12.0)
  grids required: 88
  grids cached:   88


## Compute

Which resources this run needs depends on where its numbers come from, so the decision is made here
rather than assumed. A `full` run reduces every day of the fine-grid, coarse-grid, and raw arrays, so
it wants a distributed [Coiled](https://coiled.io) cluster in `us-west-2` alongside the data. A
`cached` run reads about 120 MB of committed grids and does its arithmetic in the kernel, so starting
a cluster for it would cost money and wall-clock time for nothing.

| Mode | Reads | Cluster | Wall time | Typical use |
| --- | --- | --- | --- | --- |
| `full` | ~1.6 TB of source data | 30 x `r8g.8xlarge` | 9 min 42 s measured | once per store version, to produce the cache |
| `cached` | ~120 MB of committed grids | none | seconds | every investigation and tolerance change afterwards |

`auto` resolves to `cached` whenever every required grid is committed. This cell sits after the cache
cell because the decision needs to see what is on the branch, and an explicit `cached` against an
incomplete cache raises here rather than falling through to a terabyte-scale reduction on a kernel
with nothing behind it.

The cluster settings mirror `srm.config.ClusterConfig`, the configuration the pipeline itself uses,
but pin a *fixed*, *on-demand* worker pool rather than an adaptive spot one. These are long reductions
over the full record, and a worker that disappears mid-run through spot reclamation or an adaptive
downscale discards partials that `frisky` cannot recompute.

In [9]:
if COMPUTE_MODE not in {"auto", "full", "cached"}:
    raise ValueError(f"COMPUTE_MODE must be 'auto', 'full', or 'cached', got {COMPUTE_MODE!r}")
RESOLVED_MODE = ("cached" if not _missing else "full") if COMPUTE_MODE == "auto" else COMPUTE_MODE

if RESOLVED_MODE == "cached" and _missing:
    raise RuntimeError(
        f"COMPUTE_MODE='cached' but {len(_missing)} grid(s) are not committed, first few: "
        f"{[mean_group(key) for key in _missing[:3]]}\n"
        f"Run this notebook once with COMPUTE_MODE='full' on branch {BRANCH} to produce them."
    )

print(f"COMPUTE_MODE={COMPUTE_MODE!r} -> {RESOLVED_MODE!r}")
if _missing:
    print(f"  {len(_missing)} of {len(REQUIRED_KEYS)} grids missing, so this run recomputes them")

if RESOLVED_MODE == "full":
    # Prefer fewer, larger worker processes with more threads each (coiled runs one process per VM,
    # threads = vCPUs): the reductions are GIL-releasing zarr/numpy and the intermediates are shared
    # within a process. Use a FIXED, on-demand pool, not adaptive spot, so no worker vanishes
    # mid-reduction and discards partials frisky surfaces as an unrecoverable invariant violation.
    cfg = ClusterConfig(
        n_workers=30,
        worker_vm_types=["r8g.8xlarge"],
        scheduler_vm_types=["r8g.xlarge"],
        spot_policy="on-demand",
    )
    cluster = coiled.Cluster(
        name="srm-qaqc-trend-distortion",
        region=cfg.region,
        n_workers=cfg.n_workers,
        worker_vm_types=cfg.worker_vm_types,
        scheduler_vm_types=cfg.scheduler_vm_types,
        spot_policy=cfg.spot_policy,
        use_best_zone=True,
        tags=cfg.tags,
        worker_disk_size=250,
    )
    client = hijack(cluster.get_client())
else:
    # Cached runs read megabytes and reduce nothing, so the kernel's own threads are enough. Leaving
    # these None is what lets the shutdown cell at the end stay honest about having nothing to tear
    # down.
    cluster = client = None
    dask.config.set(scheduler="threads")
    print("  no cluster created; using the kernel's threaded scheduler")

client

COMPUTE_MODE='auto' -> 'cached'
  no cluster created; using the kernel's threaded scheduler


## Part 1: the period-mean grids

Each leaf is collapsed to one grid per pipeline stage: historical leaves averaged over `SLICE_HIST`,
scenario leaves over `SLICE_FUT`. The downscaled output is on the fine grid while raw and coarse
debiased are on the coarse GCM grid, so `interpolate_fine_to_coarse_grid` conservatively recoarsens
each fine mean onto a single fixed raw-grid template. That recoarsened grid, not the fine one, is
what the distortion comparisons use, and it is why a fine-grid downscaling effect can be separated
from a coarse-grid debiasing effect later instead of the two being conflated.

The grids are materialized one variable at a time. Batching that way bounds peak cluster memory and
graph size to a single variable, lets a failed batch cost one variable instead of the run, and gives
each variable its own commit. `lazy_period_mean` is memoized so the recoarsened grid and the fine
grid it derives from share one graph, which is what keeps the fine reduction from running twice.

In [10]:
CESM = catalog.get(GCM).to_xarray()  # DataTree keyed by scenario group

# One fixed coarse target grid for recoarsening. Using a single template rather than a per-leaf
# reference keeps every recoarsened grid on identical coordinates, so the deltas subtract cell for
# cell without xarray having to align two grids that differ in floating-point noise.
COARSE_GRID = as_grid(CESM["historical"]["tas"].isel(time=0, ensemble_member=0, drop=True))
WEIGHTS = area_weights(COARSE_GRID["lat"])


def window_for(scenario: str) -> slice:
    """Historical leaves average over the baseline window, scenario leaves over the future one."""
    return SLICE_HIST if scenario == "historical" else SLICE_FUT


def source_array(stage: str, scenario: str, var: str, member: str) -> xr.DataArray:
    """The daily source series behind one stage of one leaf."""
    if stage == "raw":
        return CESM[scenario][var].sel(ensemble_member=member)
    root = "debiased_coarse/" if stage == "coarse_debiased" else ""
    return tree[f"{root}{scenario}"][f"{var}/{member}"].dataset[var]


@functools.cache
def lazy_period_mean(key: tuple[str, str, str, str]) -> xr.DataArray:
    """Lazy period-mean grid for one (stage, scenario, variable, member), in stored units.

    The memoization is on the lazy graph, so it dedupes within a single dask.compute and carries
    nothing across a kernel restart. It matters here because the recoarsened grid is built from the
    fine grid, and without a shared object the ~129 GB fine reduction would appear twice in the batch.
    """
    stage, scenario, var, member = key
    if stage == "coarsened_downscaled_debiased":
        fine = lazy_period_mean(("downscaled_debiased", scenario, var, member))
        return as_grid(interpolate_fine_to_coarse_grid(fine, COARSE_GRID))
    da = apply_subset(source_array(stage, scenario, var, member))
    return as_grid(da.sel(time=window_for(scenario)).mean("time"))


def compute_with_retry(*collections, attempts=3, label=""):
    """dask.compute with retries so a transient scheduler glitch does not abort the whole run."""
    for attempt in range(1, attempts + 1):
        try:
            return dask.compute(*collections)
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"    {label}retry {attempt}/{attempts - 1} after {type(exc).__name__}")
            time.sleep(5)

In [11]:
%%time

means: dict[tuple[str, str, str, str], xr.DataArray] = {}
failed_vars: list[str] = []

# A cached run has nothing to compute. The committed grids are the same arrays the batch below would
# produce, so everything downstream works from them unchanged and without a cluster.
if RESOLVED_MODE == "cached":
    means = load_period_means(REQUIRED_KEYS)
    print(f"cached mode: loaded {len(means)} grid(s) from {CACHE_STORE_URI} ({BRANCH})")

for var in VARIABLES if RESOLVED_MODE == "full" else []:
    batch_keys = [key for key in REQUIRED_KEYS if key[2] == var]
    if not batch_keys:
        print(f"  {var}: no enumerated comparison needs it, skipping")
        continue
    try:
        (computed,) = compute_with_retry(
            {key: lazy_period_mean(key) for key in batch_keys}, label=f"[{var}] "
        )
    except Exception as exc:
        failed_vars.append(var)
        print(f"  {var}: FAILED after retries ({type(exc).__name__}); continuing with the rest")
        continue
    means.update(computed)
    snapshot = save_period_means(computed, var)
    print(f"  {var}: {len(computed)} grid(s) computed and committed ({snapshot[:12]})")

if failed_vars:
    print(f"\n!! variables that did not compute: {failed_vars} -- rerun before trusting results")

# Drop comparisons whose grids are not all present, so a partial run reports on what it has rather
# than raising halfway through the summary below.
_ready = comparisons[
    [
        all(
            (stage, scenario, row.variable, member) in means
            for stage in DISTORTION_STAGES
            for scenario, member in (
                (row.after_scenario, row.after_member),
                (row.before_scenario, row.before_member),
            )
        )
        for row in comparisons.itertuples()
    ]
]
if len(_ready) < len(comparisons):
    dropped = sorted(set(comparisons.comparison_id) - set(_ready.comparison_id))
    print(f"\n{len(dropped)} comparison(s) missing grids and dropped: {dropped}")
comparisons = _ready
print(f"\n{len(means)} grid(s) available, covering {len(comparisons)} comparison(s)")

cached mode: loaded 88 grid(s) from s3://carbonplan-scratch/srm/qaqc/trend-CESM2-WACCM-ERA5-global.icechunk (v0.12.0)

88 grid(s) available, covering 22 comparison(s)
CPU times: user 1.36 s, sys: 388 ms, total: 1.75 s
Wall time: 18.8 s


## Part 2: deltas, distortion, and the summary table

From here on nothing touches S3 and nothing needs a cluster. Each comparison's four stage grids are
scaled into reporting units, differenced to give absolute and percent deltas, and then differenced
again across stages to give the distortion. The both-tolerance rule turns that into a boolean mask,
and `distortion_summary` reduces each mask and field to one row.

The flagged fraction is weighted by `cos(lat)`. A plain spatial mean would give a cell at 89 degrees
the same weight as one at the equator despite covering roughly a sixtieth of the area, so an
unweighted fraction overstates whatever happens near the poles. The denominator is the valid area
rather than the whole grid, because conservative recoarsening can leave NaN cells at the grid edge
and counting those as "not distorted" would deflate every fraction by the width of that band.

Reading the absolute and percent columns together matters for `pr` in particular, because debiasing
changes the *magnitude* of precipitation and not only its trend. Where the raw GCM is too dry, the
debiased mean is substantially larger, so a scenario change that is preserved in relative terms is
still a much larger change in mm/yr. The drill-down at the end of the notebook shows this directly at
one cell, and it is the main reason the `pr` absolute distortions run into the hundreds of mm/yr.

In [12]:
def stage_grids(scenario: str, var: str, member: str) -> dict[str, xr.DataArray]:
    """The four stage grids for one leaf, converted to the variable's reporting units."""
    scale = settings(var)["scale"]
    return {stage: means[(stage, scenario, var, member)] * scale for stage in DISTORTION_STAGES}


def comparison_result(row) -> dict:
    """Deltas, distortion fields, flags, and summary rows for one comparison."""
    after = stage_grids(row.after_scenario, row.variable, row.after_member)
    before = stage_grids(row.before_scenario, row.variable, row.before_member)
    deltas, deltas_pct = compute_deltas(after, before)

    fields, flags, rows = {}, {}, []
    for stage_pair, (tested, reference) in DISTORTION_STAGE_PAIRS.items():
        absolute, percent = distortion_fields(deltas, deltas_pct, stage_pair)
        abs_tol, pct_tol = tolerances(row.variable, stage_pair)
        flag = calculate_distortion_flags(
            absolute, percent, tolerance_absolute=abs_tol, tolerance_pct=pct_tol
        )
        stats = distortion_summary(
            absolute,
            percent,
            flag,
            weights=WEIGHTS,
            sign_flip=sign_flip_mask(
                deltas[tested], deltas[reference], threshold=settings(row.variable)["sign_flip"]
            ),
        )
        fields[stage_pair] = (absolute, percent)
        flags[stage_pair] = flag
        rows.append(
            {
                "comparison_id": row.comparison_id,
                "family": row.family,
                "variable": row.variable,
                "after": f"{row.after_scenario}/{row.after_member}",
                "before": f"{row.before_scenario}/{row.before_member}",
                "stage_pair": stage_pair,
                **stats,
                "tol_abs": abs_tol,
                "tol_pct": pct_tol,
                "units": settings(row.variable)["units"],
            }
        )
    return {
        "deltas": deltas,
        "deltas_pct": deltas_pct,
        "fields": fields,
        "flags": flags,
        "rows": rows,
    }


results = {row.comparison_id: comparison_result(row) for row in comparisons.itertuples()}
summary = pd.DataFrame([row for result in results.values() for row in result["rows"]])
summary["pass"] = summary.frac_area_distorted < FLAG_THRESHOLD

_display = summary.drop(columns=["comparison_id"]).copy()
for column in ["frac_area_distorted", "frac_area_sign_flip"]:
    _display[column] = _display[column].map("{:.4%}".format)
for column in [
    "distortion_abs_max",
    "distortion_abs_min",
    "distortion_pct_max",
    "distortion_pct_min",
]:
    _display[column] = _display[column].round(3)
print(_display.to_string(index=False))
print(
    f"\nStage pairs within tolerance (< {FLAG_THRESHOLD:.0%} of weighted area flagged): "
    f"{int(summary['pass'].sum())} of {len(summary)}"
)

  family variable       after              before                             stage_pair frac_area_distorted  distortion_abs_max  distortion_abs_min  distortion_pct_max  distortion_pct_min frac_area_sign_flip  tol_abs  tol_pct units  pass
ssp_hist     hurs  ssp245/003 historical/r3i1p1f1                 coarse_debiased_vs_raw             1.0564%               6.585              -4.353               5.984              -8.935             0.0000%     1.00     2.00     %  True
ssp_hist     hurs  ssp245/003 historical/r3i1p1f1             downscaled_debiased_vs_raw             1.1378%               6.728              -3.333               6.129              -6.017             0.0143%     1.00     2.00     %  True
ssp_hist     hurs  ssp245/003 historical/r3i1p1f1 downscaled_debiased_vs_coarse_debiased             0.1127%               2.987              -1.682               5.334              -2.986             0.0010%     1.00     0.50     %  True
ssp_hist     hurs  ssp245/008 historical/r3i

In [13]:
pathlib.Path(QAQC_DIR).mkdir(parents=True, exist_ok=True)

summary.to_csv(f"{QAQC_DIR}trend_distortion_summary.csv", index=False)
for comparison_id, group in summary.groupby("comparison_id"):
    group.to_csv(f"{QAQC_DIR}trend_distortion_{comparison_id}.csv", index=False)
print(f"Wrote 1 combined and {summary.comparison_id.nunique()} per-comparison CSV(s) to {QAQC_DIR}")

summary.pivot_table(
    index=["family", "variable", "after", "before"],
    columns="stage_pair",
    values="frac_area_distorted",
)

Wrote 1 combined and 22 per-comparison CSV(s) to /scratch/synced/qa_plots/trend_distortion/


stage_pair                                         coarse_debiased_vs_raw  \
family   variable after       before                                        
g6_hist  hurs     g6_1p5k/003 historical/r3i1p1f1                0.006143   
         pr       g6_1p5k/003 historical/r3i1p1f1                0.261615   
         rsds     g6_1p5k/003 historical/r3i1p1f1                0.687242   
         tas      g6_1p5k/003 historical/r3i1p1f1                0.998218   
         tasmax   g6_1p5k/003 historical/001                     0.998176   
         tasmin   g6_1p5k/003 historical/001                     0.999474   
g6_ssp   hurs     g6_1p5k/003 ssp245/003                         0.001119   
         pr       g6_1p5k/003 ssp245/003                         0.256806   
         rsds     g6_1p5k/003 ssp245/003                         0.574337   
         tas      g6_1p5k/003 ssp245/003                         0.997190   
         tasmax   g6_1p5k/003 ssp245/008                         0.998019   
         tasmin   g6_1p5k/003 ssp245/008                         0.999365   
ssp_hist hurs     ssp245/003  historical/r3i1p1f1                0.010564   
                  ssp245/008  historical/r3i1p1f1                0.008429   
         pr       ssp245/003  historical/r3i1p1f1                0.339808   
                  ssp245/008  historical/r3i1p1f1                0.325932   
         rsds     ssp245/003  historical/r3i1p1f1                0.694342   
                  ssp245/008  historical/r3i1p1f1                0.668430   
         tas      ssp245/003  historical/r3i1p1f1                0.997993   
                  ssp245/008  historical/r3i1p1f1                0.998131   
         tasmax   ssp245/008  historical/001                     0.998593   
         tasmin   ssp245/008  historical/001                     0.999348   

stage_pair                                         downscaled_debiased_vs_coarse_debiased  \
family   variable after       before                                                        
g6_hist  hurs     g6_1p5k/003 historical/r3i1p1f1                                0.001050   
         pr       g6_1p5k/003 historical/r3i1p1f1                                0.103362   
         rsds     g6_1p5k/003 historical/r3i1p1f1                                0.006592   
         tas      g6_1p5k/003 historical/r3i1p1f1                                0.993831   
         tasmax   g6_1p5k/003 historical/001                                     0.994902   
         tasmin   g6_1p5k/003 historical/001                                     0.996061   
g6_ssp   hurs     g6_1p5k/003 ssp245/003                                         0.000243   
         pr       g6_1p5k/003 ssp245/003                                         0.081127   
         rsds     g6_1p5k/003 ssp245/003                                         0.003235   
         tas      g6_1p5k/003 ssp245/003                                         0.992672   
         tasmax   g6_1p5k/003 ssp245/008                                         0.993524   
         tasmin   g6_1p5k/003 ssp245/008                                         0.995305   
ssp_hist hurs     ssp245/003  historical/r3i1p1f1                                0.001127   
                  ssp245/008  historical/r3i1p1f1                                0.001130   
         pr       ssp245/003  historical/r3i1p1f1                                0.116543   
                  ssp245/008  historical/r3i1p1f1                                0.143887   
         rsds     ssp245/003  historical/r3i1p1f1                                0.010153   
                  ssp245/008  historical/r3i1p1f1                                0.008338   
         tas      ssp245/003  historical/r3i1p1f1                                0.994299   
                  ssp245/008  historical/r3i1p1f1                                0.994359   
         tasmax   ssp245/008  historical/001                                     0.995142   
         tas

### Are the tolerances in the right place?

The `rsds`, `hurs`, and `dtr` tolerances above are reasoned rather than measured, so this table puts
the number in force next to the distortion actually observed. `tol_percentile` is the most directly
useful column: it says what share of cells the tolerance currently excludes, so a tolerance sitting
at the 99.9th percentile is flagging only the extreme tail while one at the 50th is flagging half the
map. Percentiles are taken over each comparison's own cells and then averaged across the comparisons
that share a variable and stage pair.

Adjusting a tolerance means editing `_SETTINGS_ROWS` and rerunning from Part 2, which reads the cache
and takes seconds.

What this run shows is that the tolerances are not comparably strict across variables. The
temperature ones sit far out in the tail, at the 99.7th percentile for `tas` and the 94.4th for
`tasmin`, so the absolute condition is genuinely selective, which matters because the percent
condition is disabled for them. For `pr` and `rsds` the absolute tolerance sits near the middle of the
distribution, at the 57th and 48th percentiles, so it filters almost nothing and the percent tolerance
decides the flag alone. The `downscaling_only` rows are the exception in every case, sitting between
the 96th and 99.98th percentiles, because downscaling moves the coarse-scale delta far less than
debiasing does.

In [14]:
CALIBRATION_QUANTILES = (0.5, 0.9, 0.99, 0.999)


def calibration_table() -> pd.DataFrame:
    """Observed |absolute distortion| percentiles next to the tolerance in force."""
    rows = []
    for row in comparisons.itertuples():
        for stage_pair, (absolute, _) in results[row.comparison_id]["fields"].items():
            magnitude = np.abs(np.asarray(absolute.values, dtype="float64"))
            magnitude = magnitude[np.isfinite(magnitude)]
            if magnitude.size == 0:
                continue
            abs_tol, _ = tolerances(row.variable, stage_pair)
            rows.append(
                {
                    "variable": row.variable,
                    "stage_pair": stage_pair,
                    "units": settings(row.variable)["units"],
                    "tol_abs": abs_tol,
                    "tol_percentile": float((magnitude < abs_tol).mean() * 100),
                    **{
                        f"p{q * 100:g}": float(np.quantile(magnitude, q))
                        for q in CALIBRATION_QUANTILES
                    },
                }
            )
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    return frame.groupby(["variable", "stage_pair", "units", "tol_abs"]).mean(numeric_only=True)


_calibration = calibration_table()
print(
    _calibration.round(4).to_string() if len(_calibration) else "no distortion fields to calibrate"
)

                                                               tol_percentile     p50      p90       p99     p99.9
variable stage_pair                             units tol_abs                                                     
hurs     coarse_debiased_vs_raw                 %     1.00            94.7804  0.1286   0.7259    1.4764    2.4466
         downscaled_debiased_vs_coarse_debiased %     1.00            99.8834  0.0103   0.1171    0.4539    1.0041
         downscaled_debiased_vs_raw             %     1.00            94.6628  0.1358   0.7370    1.4784    2.4400
pr       coarse_debiased_vs_raw                 mm/yr 5.00            36.6482  8.0816  42.8063  175.3154  726.6474
         downscaled_debiased_vs_coarse_debiased mm/yr 5.00            86.8218  1.1852   6.0506   19.9457   50.7281
         downscaled_debiased_vs_raw             mm/yr 5.00            36.6826  8.0591  42.8707  177.6139  718.2077
rsds     coarse_debiased_vs_raw                 W m-2 0.25            27.0671  0

## Part 3: one figure per comparison

Each comparison gets a single 2 x 3 figure: rows are absolute and percent distortion, columns are the
three stage pairs. Keeping all three columns is deliberate. Two panels would be enough to show *that*
a delta moved, but only the third column separates what debiasing did from what downscaling added on
top of it, which is the question the notebook exists to answer.

Color limits come from the 99th percentile of each panel's own magnitudes rather than a per-variable
constant. That is what lets one function serve a kelvin temperature panel, an mm/yr precipitation
panel, and a W m-2 radiation panel without a hand-maintained table of plot ranges, and it keeps a
single outlier cell from flattening the scale.

Every figure is written to `QAQC_DIR`. Only the comparisons crossing `FLAG_THRESHOLD` render inline by
default, because rendering all of them would make this notebook several times larger. On this run that
meant 22 figures saved and 9 rendered, the 9 being every `pr` and `rsds` comparison against historical
plus the two `tasmin` ones.

In [15]:
PROJ = ccrs.PlateCarree()
CBAR = {"orientation": "horizontal", "pad": 0.05, "shrink": 0.8}


def format_map(ax) -> None:
    ax.coastlines(linewidth=0.3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.2)


def robust_limit(da) -> float | None:
    """99th-percentile magnitude of the finite non-zero values, or None when there are none.

    A high quantile rather than the raw maximum keeps one outlier cell from flattening the scale, and
    None lets matplotlib autoscale when a panel is entirely zero.
    """
    values = np.abs(np.asarray(da.values, dtype="float64"))
    values = values[np.isfinite(values) & (values > 0)]
    if values.size == 0:
        return None
    return float(np.quantile(values, 0.99))


def plot_comparison(row, result: dict):
    """2 x 3 distortion figure for one comparison: absolute and percent, three stage pairs."""
    units = settings(row.variable)["units"]
    by_pair = {r["stage_pair"]: r for r in result["rows"]}
    fig, axes = plt.subplots(2, 3, figsize=(21, 9), subplot_kw={"projection": PROJ})

    for col, stage_pair in enumerate(DISTORTION_STAGE_PAIRS):
        absolute, percent = result["fields"][stage_pair]
        stats = by_pair[stage_pair]
        panels = ((absolute, f"absolute ({units})"), (percent, "percent (%)"))
        for rank, (field, label) in enumerate(panels):
            ax = axes[rank, col]
            limit = robust_limit(field)
            field.plot(
                ax=ax,
                transform=PROJ,
                cmap="RdBu_r",
                vmin=None if limit is None else -limit,
                vmax=limit,
                cbar_kwargs={**CBAR, "label": label},
            )
            format_map(ax)
            if rank == 0:
                ax.set_title(
                    f"{stage_pair}\n{stats['frac_area_distorted']:.3%} of area flagged"
                    f" (tol {stats['tol_abs']:g} {units}, {stats['tol_pct']:g} %),"
                    f" sign flip {stats['frac_area_sign_flip']:.3%}",
                    fontsize=8,
                )
            else:
                ax.set_title("")

    fig.suptitle(
        f"{row.variable}: {row.after_scenario}/{row.after_member} minus "
        f"{row.before_scenario}/{row.before_member} ({row.family})",
        fontsize=14,
    )
    fig.tight_layout()
    return fig


def should_show(comparison_id: str, flagged: bool) -> bool:
    """Whether a comparison's figure renders inline, per SHOW_FIGURES."""
    if SHOW_FIGURES == "all":
        return True
    if SHOW_FIGURES == "none":
        return False
    if SHOW_FIGURES == "flagged":
        return flagged
    return comparison_id in SHOW_FIGURES

In [ ]:
_shown, _saved = 0, 0
for row in comparisons.itertuples():
    result = results[row.comparison_id]
    flagged = any(r["frac_area_distorted"] >= FLAG_THRESHOLD for r in result["rows"])
    fig = plot_comparison(row, result)
    if SAVE_FIGURES:
        fig.savefig(f"{QAQC_DIR}distortion_{row.comparison_id}.png", dpi=100, bbox_inches="tight")
        _saved += 1
    if should_show(row.comparison_id, flagged):
        plt.show()
        _shown += 1
    else:
        plt.close(fig)

print(
    f"{_saved} figure(s) saved to {QAQC_DIR}, {_shown} rendered inline (SHOW_FIGURES={SHOW_FIGURES!r})"
)

## All three families in one figure

One figure per variable in `OVERVIEW_VARIABLES`, for scanning which family and which regions carry
the largest distortion before going back to the per-comparison figures for detail. Columns are the
three scenario families and rows are the raw delta, the downscaled delta on its own fine grid, and
the distortion between them.

Where a family covers a variable with more than one member, the first enumerated comparison is shown
and its members are named in the panel title. None of these panels reflect ensemble spread, so a
distortion visible for one member's realization of internal variability need not appear for another.

In [ ]:
def plot_overview(var: str):
    """Raw delta, downscaled delta, and their difference, across the scenario families."""
    picked = {}
    for family in DISTORTION_FAMILIES:
        rows = comparisons[(comparisons.family == family) & (comparisons.variable == var)]
        if len(rows):
            picked[family] = rows.iloc[0]
    if not picked:
        print(f"no enumerated comparison covers {var}")
        return None

    entry = settings(var)
    fig, axes = plt.subplots(
        3,
        len(picked),
        figsize=(7 * len(picked), 12),
        subplot_kw={"projection": PROJ},
        squeeze=False,
    )
    for col, (family, row) in enumerate(picked.items()):
        deltas = results[row.comparison_id]["deltas"]
        panels = (
            (deltas["raw"], "raw delta"),
            (deltas["downscaled_debiased"], "downscaled delta (fine grid)"),
            (deltas["coarsened_downscaled_debiased"] - deltas["raw"], "downscaled minus raw"),
        )
        for rank, (field, label) in enumerate(panels):
            ax = axes[rank, col]
            limit = robust_limit(field)
            field.plot(
                ax=ax,
                transform=PROJ,
                cmap=entry["cmap"],
                vmin=None if limit is None else -limit,
                vmax=limit,
                cbar_kwargs={**CBAR, "label": entry["units"]},
            )
            format_map(ax)
            ax.set_title(
                f"{label}: {family}\n{row.after_member} minus {row.before_member}", fontsize=9
            )

    fig.suptitle(f"{var} period-mean delta and distortion ({entry['units']})", fontsize=15)
    fig.tight_layout()
    return fig


for _var in OVERVIEW_VARIABLES:
    _fig = plot_overview(_var)
    if _fig is None:
        continue
    if SAVE_FIGURES:
        _fig.savefig(f"{QAQC_DIR}overview_{_var}.png", dpi=100, bbox_inches="tight")
    plt.show()

## Part 4: single-point drill-down

The maps show *where* a delta moved; this section looks at *why*, for one grid cell across the daily
record. Both scenarios of the chosen comparison are plotted at each of the three pipeline stages, so
a distortion can be traced to the stage that introduced it.

This is the one section the cache does not make cheap, and the reason is chunking rather than volume.

| Series | Chunking | Read for one point |
| --- | --- | --- |
| downscaled, coarse debiased | `(time 8000, lat 8, lon 16)` | ~16 MB each |
| raw GCM | `(member 4, time 120, lat 192, lon 288)` | ~10 GB each |

The raw store keeps the full spatial extent in every chunk, so selecting a single column still reads
every latitude and longitude in each time chunk it touches. Two raw series is roughly 20 GB. All
series are built lazily and resolved in one `dask.compute` so those reads overlap instead of queueing,
which is why the cell below takes seconds with the cluster up. In a `cached` run there is no cluster
and the same reads fall to the kernel's threads, so expect minutes rather than seconds for a point
that has not been looked at yet; narrowing `SLICE_FUT` is the simplest way to make one cheaper.

In [ ]:
# Which comparison and which cell to inspect. None picks the worst-flagged comparison in the summary.
DRILL_COMPARISON = None
DRILL_LAT, DRILL_LON = -21.006, -56.377  # Brazil, near the Paraguay border

if DRILL_COMPARISON is None:
    _ranked = summary.sort_values("frac_area_distorted", ascending=False)
    DRILL_COMPARISON = str(_ranked.iloc[0]["comparison_id"])
# Select by mask rather than set_index().loc: the latter drops comparison_id into the index, and the
# figure title reads it off the row.
_match = comparisons[comparisons.comparison_id == DRILL_COMPARISON]
if _match.empty:
    raise KeyError(f"{DRILL_COMPARISON!r} is not an enumerated comparison; see the table above")
drill_row = _match.iloc[0]
drill_entry = settings(drill_row.variable)

print(f"Drilling into {DRILL_COMPARISON} at lat={DRILL_LAT}, lon={DRILL_LON}")
print(summary[summary.comparison_id == DRILL_COMPARISON].to_string(index=False))


def point_series(row, lat: float, lon: float) -> dict[str, xr.DataArray]:
    """Every daily series the drill-down plots for one grid cell, lazy and in stored units.

    Nothing is computed here. Pass the whole dict to a SINGLE dask.compute so every read shares one
    graph and one scheduler round-trip, rather than paying a round-trip per series with the workers
    idle in between.
    """
    at_point = {"lat": lat, "lon": lon, "method": "nearest"}
    series = {}
    for side, scenario, member in (
        ("after", row.after_scenario, row.after_member),
        ("before", row.before_scenario, row.before_member),
    ):
        for stage in ("raw", "coarse_debiased", "downscaled_debiased"):
            da = source_array(stage, scenario, row.variable, member)
            series[f"{stage}/{side}"] = da.sel(**at_point).sel(time=window_for(scenario))
    return series

In [ ]:
%%time
(drill_series,) = dask.compute(point_series(drill_row, DRILL_LAT, DRILL_LON))
print(f"Resolved {len(drill_series)} series in one compute")

In [ ]:
def plot_point_panels(series: dict, row, lat: float, lon: float):
    """Day-of-year means for both scenarios at each pipeline stage, one panel per stage.

    Daily values carry the daily reporting factor, not the annual one used for period means. For `pr`
    that is mm/day rather than mm/yr, a 365-fold difference that would otherwise pass unnoticed.
    """
    entry = settings(row.variable)
    scale, units = entry["daily_scale"], entry["daily_units"]
    stages = ("raw", "coarse_debiased", "downscaled_debiased")
    fig, axes = plt.subplots(1, len(stages), figsize=(6 * len(stages), 5), sharey=True)

    for ax, stage in zip(axes, stages, strict=True):
        for side, label, color in (
            ("after", f"{row.after_scenario}/{row.after_member}", "darkorange"),
            ("before", f"{row.before_scenario}/{row.before_member}", "crimson"),
        ):
            doy = (series[f"{stage}/{side}"] * scale).groupby("time.dayofyear").mean()
            ax.plot(doy["dayofyear"], doy, lw=1.2, color=color, label=label)
        ax.set_title(stage)
        ax.set_xlabel("Day of year")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel(f"{row.variable} ({units})")
    axes[-1].legend(fontsize=8)

    fig.suptitle(f"{row.comparison_id} at lat={lat}, lon={lon}", fontsize=13)
    fig.tight_layout()
    return fig


_fig = plot_point_panels(drill_series, drill_row, DRILL_LAT, DRILL_LON)
if SAVE_FIGURES:
    _fig.savefig(f"{QAQC_DIR}point_{DRILL_COMPARISON}.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# The numeric version of the panels above: this point's period means and delta at each stage, in the
# same reporting units as the summary table, plus the distortion each stage pair reports here.
_scale = drill_entry["scale"]
_point_means = {
    stage: {
        side: float((drill_series[f"{stage}/{side}"] * _scale).mean())
        for side in ("after", "before")
    }
    for stage in ("raw", "coarse_debiased", "downscaled_debiased")
}
_rows = []
for stage, sides in _point_means.items():
    delta = sides["after"] - sides["before"]
    _rows.append(
        {
            "stage": stage,
            "after": round(sides["after"], 3),
            "before": round(sides["before"], 3),
            "delta": round(delta, 3),
            "delta_pct": round(delta * 100 / sides["before"], 3) if sides["before"] else np.nan,
        }
    )
_point_table = pd.DataFrame(_rows)
print(f"Period means at ({DRILL_LAT}, {DRILL_LON}) in {drill_entry['units']}:")
print(_point_table.to_string(index=False))

_raw_delta = _point_table.set_index("stage").loc["raw", "delta"]
for stage in ("coarse_debiased", "downscaled_debiased"):
    _stage_delta = _point_table.set_index("stage").loc[stage, "delta"]
    print(
        f"  {stage} minus raw distortion: {_stage_delta - _raw_delta:+.3f} {drill_entry['units']}"
    )

## Summary

Every comparison is measured three ways and reduced to one row apiece: the area-weighted fraction of
the domain where the delta moved by more than both tolerances, the signed extremes of that movement,
the fraction where two stages disagree on the sign of the change, and the tolerances that produced the
flag. On the v0.12.0 store that is 22 comparisons and 66 rows, of which 44 stay within the 2 % area
test.

The worst flagged fraction per variable and family, across the three stage pairs:

| Variable | SSP245 minus historical | G6 minus historical | G6 minus SSP245 |
| --- | --- | --- | --- |
| `tas` | 0.41 % | 0.10 % | 0.02 % |
| `tasmax` | 1.41 % | 0.65 % | 0.18 % |
| `tasmin` | 5.32 % | 2.44 % | 0.75 % |
| `hurs` | 1.62 % | 1.26 % | 0.57 % |
| `rsds` | 6.44 % | 4.71 % | 1.41 % |
| `pr` | 29.54 % | 22.77 % | 22.45 % |

Two results stand out. The first is that **debiasing accounts for nearly all of the distortion and
downscaling adds very little on top of it.** For `pr` against historical the delta moves on 29.5 % of
the area between raw and coarse debiased, and on 29.2 % between raw and downscaled, but only 5.8 %
between coarse debiased and downscaled. The same ordering holds for every variable: `rsds` goes 6.1 %,
6.4 %, 1.0 %, and `tas` goes 0.30 %, 0.41 %, 0.07 %. Splitting the comparison three ways is what makes
that visible, and it points any follow-up at the quantile mapping rather than at the spatial
disaggregation.

The second is that the variables separate cleanly by how much they are affected. Temperature is
essentially untouched, and no temperature comparison flips the sign of its delta anywhere on the globe.
`pr` is affected everywhere, on a fifth to a third of the area, with absolute distortions reaching
1676 mm/yr. For the SAI comparison specifically, the one these outputs exist to inform, debiasing
reverses the sign of the precipitation response on 2.7 % of the area, and `rsds` on 0.34 %. Those
sign-flip fractions are the most decision-relevant numbers here, because a flipped cell is one where
"does SAI make this place wetter or drier" depends on which pipeline stage you read.

What changed relative to the hand-configured version of this notebook:

| Before | After |
| --- | --- |
| one variable, three hardcoded members | every leaf the store carries, baselines resolved from `srm.lineage` |
| nine period-mean grids, recomputed every session | 88 grids committed to an icechunk cache, ~120 MB for a ~1.6 TB read |
| roughly fourteen figures, none saved | one figure per comparison, 22 saved, 9 flagged ones rendered |
| no summary output | 66 CSV rows plus a tolerance calibration table |
| thresholds for two variable families | thresholds for six, with the calibration table saying where each one sits |

The drill-down explains much of the `pr` result. At the example cell in Brazil the period-mean
precipitation is 962 mm/yr raw, 1440 coarse debiased, and 1677 downscaled, so debiasing raises the
magnitude by roughly three quarters. The scenario change at that cell is -4.65 % raw and -4.09 %
coarse debiased, which is close agreement in relative terms, yet the same change reads as -47 mm/yr
raw against -61 mm/yr debiased. The absolute distortion of -14 mm/yr is therefore mostly a consequence
of correcting a dry bias, not of the pipeline changing the trend. That is worth keeping in mind before
treating a 29 % flagged area for `pr` as a defect: the percent columns and the sign-flip fraction are
the better guides for precipitation, and the sign-flip fraction stays near 2.5 %.

Two caveats. The check does no significance testing, so a flagged cell does not distinguish "the
pipeline changed a real signal" from "the pipeline changed what was mostly noise". Each comparison uses
one member per scenario, so none of these deltas reflect ensemble spread.

In [27]:
if cluster is not None:
    cluster.shutdown()
else:
    print("no cluster was created (cached run); nothing to shut down")

no cluster was created (cached run); nothing to shut down
